# Step 3: Create GitHub Environments

![GitHub](https://img.shields.io/badge/GitHub-181717?logo=github&logoColor=white)
![GitHub CLI](https://img.shields.io/badge/GitHub_CLI-181717?logo=github&logoColor=white)

This notebook creates GitHub environments (dev, staging, production) in each repository and optionally adds production protection rules.

**What this notebook does:**
1. Loads configuration from `config.json` (or collects it fresh)
2. Fetches the repository list
3. Creates environments via the GitHub API
4. Optionally adds required reviewers to the production environment

> **Prerequisite:** Run **[02_setup_credentials.ipynb](02_setup_credentials.ipynb)** first, or at least have `config.json` with your org name and repo list.

## Prerequisites Check

In [ ]:
import subprocess, json, sys, os

def run_cmd(cmd, capture=True, check=True):
    """Run a shell command and return the result."""
    result = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and result.returncode != 0:
        print(f"ERROR: {result.stderr.strip()}")
        return None
    return result

# Check GitHub CLI
r = run_cmd("gh --version", check=False)
if r and r.returncode == 0:
    print(f"✓ GitHub CLI installed: {r.stdout.strip().splitlines()[0]}")
else:
    print("✗ GitHub CLI not found. Install: https://cli.github.com")
    sys.exit(1)

# Check GitHub authentication
r = run_cmd("gh auth status", check=False)
if r and r.returncode == 0:
    print("✓ GitHub CLI is authenticated.")
else:
    print("✗ Not authenticated. Run 'gh auth login' in a terminal first.")
    sys.exit(1)

print("\n✓ All prerequisites met.")

## Load Configuration

In [ ]:
# Load config from previous notebooks
config_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "config.json")
config = {}
if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    print(f"✓ Loaded config from {config_path}")
else:
    print("No config.json found. You'll enter values manually.")

ORG_NAME = config.get("org_name") or input("GitHub organization name: ").strip()
ENVIRONMENTS = config.get("environments", ["dev", "staging", "production"])
REPOS = config.get("repos", [])

if not REPOS:
    mode = input("No repos in config. Fetch from GitHub? (y/n): ").strip().lower()
    if mode == "y":
        r = run_cmd(f"gh repo list {ORG_NAME} --limit 1000 --json name --jq '.[].name'", check=False)
        if r and r.returncode == 0:
            REPOS = [line.strip() for line in r.stdout.strip().split("\n") if line.strip()]
        else:
            print(f"✗ Failed to fetch repos: {r.stderr.strip() if r else ''}")
    else:
        repos_input = input("Enter repo names (comma-separated): ").strip()
        REPOS = [r.strip() for r in repos_input.split(",") if r.strip()]

print(f"\nOrganization: {ORG_NAME}")
print(f"Environments: {ENVIRONMENTS}")
print(f"Repositories: {len(REPOS)}")

## Configure Production Protection (Optional)

Optionally set a required reviewer for the production environment. Leave blank to skip.

In [ ]:
# --- Production Protection Rules ---
# Enter your GitHub numeric user ID (find via: gh api /user --jq '.id')
# Leave blank to skip production protection rules.

PROD_REVIEWER_USER_ID = input("Production reviewer user ID (blank to skip): ").strip()

if PROD_REVIEWER_USER_ID:
    print(f"✓ Production reviewer: user ID {PROD_REVIEWER_USER_ID}")
else:
    print("⊘ Skipping production protection rules.")

## Create Environments

Creates dev, staging, and production environments in each repository via the GitHub API.

In [ ]:
# --- Create GitHub Environments ---

results = []

for repo in REPOS:
    print(f"\nSetting up environments for {repo}...")

    for env in ENVIRONMENTS:
        cmd = (
            f'gh api --method PUT '
            f'-H "Accept: application/vnd.github+json" '
            f'"/repos/{ORG_NAME}/{repo}/environments/{env}" '
            f'-f "wait_timer=0"'
        )
        r = run_cmd(cmd, check=False)
        output = (r.stdout + r.stderr) if r else ""

        if r and r.returncode == 0:
            status = "✓ Created"
            print(f"  ✓ Created environment: {env}")
        else:
            status = "✗ Failed"
            print(f"  ✗ Failed: {env}")
            print(f"    {output.strip()[:200]}")

        results.append({"repo": repo, "env": env, "status": status})

        # Add production protection rules
        if env == "production" and PROD_REVIEWER_USER_ID:
            prot_cmd = (
                f'gh api --method PUT '
                f'-H "Accept: application/vnd.github+json" '
                f'"/repos/{ORG_NAME}/{repo}/environments/production" '
                f'-f "prevent_self_review=true" '
                f'-F "reviewers[][type]=User" '
                f'-F "reviewers[][id]={PROD_REVIEWER_USER_ID}"'
            )
            pr = run_cmd(prot_cmd, check=False)
            if pr and pr.returncode == 0:
                print(f"  ✓ Added protection rules for production")
            else:
                prot_output = (pr.stdout + pr.stderr) if pr else ""
                print(f"  ✗ Failed to add protection rules")
                print(f"    {prot_output.strip()[:200]}")

## Results Table

In [ ]:
# --- Display results ---

header = f"{'Repo':<30} {'Environment':<15} {'Status':<15}"
print(header)
print("-" * len(header))
for r in results:
    print(f"{r['repo']:<30} {r['env']:<15} {r['status']:<15}")

print(f"\nEnvironment setup complete!")

## Next Steps

Proceed to **[04_configure_secrets_and_workflow.ipynb](04_configure_secrets_and_workflow.ipynb)** to set up GitHub organization secrets and deploy the workflow template.